In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
food_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(food_path)

print(f"Dataset shape: {df.shape}")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 6))
plt.hist(df['Delivery_Time'], bins=60, edgecolor='black')
plt.title('Distribution of delivery_time')
plt.xlabel('delivery_time')
plt.ylabel('Count')
plt.show()

In [ ]:
df

In [ ]:
# Task 1: Write your code here:
df = df.drop("Order_ID", axis=1)

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mode()[0])

df = df.dropna(subset=['Delivery_Time'])

print("\n\nMissing now : \n")
check_missing_values(df)


In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le
print("Done")

In [ ]:
# Task 5: Write your code here:

feature_cols = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])
df.head()


In [ ]:
# Task 6: Write your code here:

#implance happen in claasification only in catogiral

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
n_splits = 5

mae_scores = []

kf = KFold(n_splits= n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]




    # Train
  model.fit(X_train, y_train)

    # Predict and evaluate
  y_pred = model.predict(X_test)

   # Calculate metrics evaluate
  mae_scores.append(mean_absolute_error(y_test, y_pred))

    # Store results
mae_scores = np.array(mae_scores)
print(f"\nMAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
# Plot feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('Distribution of delivery time')
plt.xlabel('delivery time')
plt.ylabel('Count')
plt.show()

In [ ]:
!pip install catboost

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

In [ ]:
# Task Bonus: Write your code here:

models = {

  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),

  "CatBoost": CatBoostRegressor(verbose=0)
}
all_results = {}
for name in models:
  all_results[name] = {'mae': [], 'rmse': [], 'r2': []}

for name in sklearn_models:
  all_results[name] = {'mae': [], 'rmse': [], 'r2': []}
  kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)


    # Store results
    all_results[model_name]["mae"].append(mae)

for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mae']):.4f}")
